In [12]:
import pandas as pd

df_english = pd.read_csv('./cleaned_songs_with_lyrics.csv')

In [13]:
df_english[['track_genre','id']].groupby(by=["track_genre"]).count()

,id
track_genre,
afrobeat,96
alt-rock,65
alternative,75
ambient,124
anime,73
...,...
spanish,13
synth-pop,517
techno,82


In [14]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from textblob import TextBlob # TextBlob doesn't require a separate download for basic sentiment



# 2. Extract Sentiment (Polarity: -1 to 1, Subjectivity: 0 to 1)
def get_sentiment(text):
    blob = TextBlob(str(text))
    return blob.sentiment.polarity, blob.sentiment.subjectivity

df_english[['polarity', 'subjectivity']] = df_english['lyrics_cleaned'].apply(
    lambda x: pd.Series(get_sentiment(x))
)

In [15]:
counts = df_english['track_genre'].value_counts()

# Filter for genres with 10+ members
df_english = df_english[df_english['track_genre'].isin(counts[counts >= 10].index)].copy()

In [16]:
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import LabelEncoder

# 1. Encode genres into numbers (Neural Networks prefer this)
le = LabelEncoder()
df_english['genre_label'] = le.fit_transform(df_english['track_genre'])

# 2. Advanced Pipeline
nn_pipeline = Pipeline([
    ('preprocessor', ColumnTransformer([
        # ngram_range=(1,2) lets the model see pairs of words
        ('text', TfidfVectorizer(max_features=5000, stop_words='english', ngram_range=(1,2)), 'lyrics_cleaned'),
        ('num', StandardScaler(), ['polarity', 'subjectivity'])
    ])),
    # Hidden layers of 64 and 32 neurons create complex 'decision paths'
    ('clf', MLPClassifier(hidden_layer_sizes=(64, 32), 
                          activation='relu', 
                          solver='adam', 
                          max_iter=500,
                          early_stopping=True))
])
X = df_english[['lyrics_cleaned','subjectivity','polarity']]
y = df_english['genre_label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)
# 3. Fit
nn_pipeline.fit(X_train, y_train)

# Generate class predictions (e.g., 'rock', 'pop')
y_pred = nn_pipeline.predict(X_test)

# Generate probability scores (if you want to see how confident the model is)
y_probs = nn_pipeline.predict_proba(X_test)

df_preds = pd.DataFrame({
    'lyrics': X_test.index,
    'actual_parent': y_test,
    'predicted_genre': y_pred
})

In [ ]:
df_preds['Equal?'] = df_preds['actual_parent'] == df_preds['predicted_genre']

: 

In [ ]:
# Im thinking

# We use the 97 predefined genres. 

# We do sentiment analysis to try and extract the emotions from the song

# We esentially make a matrix and cluster the genres together inter a more manageable amount of groups

# We since these clusters would have more similarity, we now ignore sentiment, and only use word based methods to try to do supervised 
# learning and define the specific genre

# Then we can analyse to see if specific emotional charge have enough variation between genres.csv

# We can also see if the spotigfy defined genres are better than out clustered ones, and see if we can use our own judgement to identify why sentiment has 
# showed these variations and if we can identify the parent genre.